In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from tdmpc2lora_tmp.config import Config
from tdmpc2lora_tmp.train_env import make_env
from tdmpc2lora_tmp.model import TDMPC2

def save_gif(frames, filename="reacher.gif", fps=30):
    """フレームリストをGIFとして保存"""
    print(f"Saving GIF to {filename} ...")
    plt.figure(figsize=(frames[0].shape[1] / 100.0, frames[0].shape[0] / 100.0), dpi=100)
    patch = plt.imshow(frames[0])
    plt.axis('off')

    def animate(i):
        patch.set_data(frames[i])

    anim = animation.FuncAnimation(plt.gcf(), animate, frames=len(frames), interval=1000/fps)
    if filename is None:
        plt.show()
    else:
        anim.save(filename, writer='pillow', fps=fps)
    plt.close()
    print("Done!")

def visualize_model(task_type, task_id, model_path, output_name):
    cfg = Config()
    cfg.task = task_type  # "SimpleReacher" or "VariableReacher"
    cfg.task_id = task_id
    cfg.num_tasks = 4
    cfg.lora_rank = 4
    cfg.device = "cpu" # 可視化はCPUでOK

    env = make_env(cfg)
    agent = TDMPC2(cfg)

    # モデルロード
    print(f"Loading: {model_path}")
    ckpt = torch.load(model_path, map_location="cpu")
    agent.model.load_state_dict(ckpt["model"], strict=False)
    agent.eval()

    # エピソード実行 & 描画
    frames = []
    obs, _ = env.reset()
    frames.append(env.env.render()) # Wrapperの奥の生Envのrenderを呼ぶ

    done = False
    total_reward = 0
    
    while not done:
        # task_idx を指定して推論
        action = agent.act(obs, eval_mode=True, task_idx=task_id)
        obs, reward, done_tensor, _ = env.step(action)
        
        done = bool(done_tensor.item())
        total_reward += reward.item()
        
        # フレーム取得
        frames.append(env.env.render())

    print(f"Episode Reward: {total_reward:.2f}")
    save_gif(frames, filename=output_name)

In [ ]:
if __name__ == "__main__":
    # 使用例: SimpleReacher の Task 0 のモデルを可視化
    # 適切なパスに変更してください
    # MODEL_PATH = None
    MODEL_PATH = "result/models/Reacher_Task0_rank4_init/best.pth"
    
    # 1. SimpleReacher Task 0 の可視化
    visualize_model("SimpleReacher", 0, MODEL_PATH, "simple_task0.gif")
    
    # 2. VariableReacher Task 1 の可視化 (もしVariableで学習したなら)
    # visualize_model("VariableReacher", 1, "path/to/variable_task1/best.pth", "variable_task1.gif")
    
    print("Visualization script ready. Adjust paths in __main__ to run.")